# GNN Shift Prediction Workflow

Dieses Notebook ermöglicht:
1. **Training** eines heterogenen GNN-Modells zur Shift-Vorhersage
2. **Vorhersage** mit einem trainierten Modell
3. **Erklärung** von Vorhersagen mittels GNNExplainer

---

## Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Projektpfade einrichten
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
EXPLAINER_DIR = SCRIPTS_DIR / "explainer"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

# Pfade zum sys.path hinzufügen
for path in [str(SCRIPTS_DIR), str(EXPLAINER_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Projekt-Root: {PROJECT_ROOT}")
print(f"Scripts-Ordner: {SCRIPTS_DIR}")
print(f"Daten-Ordner: {DATA_DIR}")
print(f"Modelle-Ordner: {MODELS_DIR}")

Projekt-Root: /home/bautz/gnn4nmr
Scripts-Ordner: /home/bautz/gnn4nmr/scripts
Daten-Ordner: /home/bautz/gnn4nmr/data
Modelle-Ordner: /home/bautz/gnn4nmr/models


In [ ]:
import random
import pickle
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Projekt-Module
from model import HeteroGNNModel
from dataloader import create_dataloaders, ShiftDataset
from train import train_model

# Für GNNExplainer
from torch_geometric.explain import Explainer, HeteroExplanation
from torch_geometric.explain.algorithm import GNNExplainer
from torch_geometric.explain.config import ModelConfig, ModelMode, ModelReturnType, ModelTaskLevel

from explainer_utils import (
    NodeTypeRegressionWrapper,
    build_dataset,
    ensure_dir,
    get_device,
    heterodata_to_dicts,
    load_config,
    load_stats,
    load_trained_model,
    summarize_incident_edges,
    summarize_node_mask,
    validate_indices,
)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'pandas'

---
## 1. Modell Training

Trainiere ein heterogenes GNN-Modell zur Vorhersage von chemischen Shifts.

### 1.1 Training Parameter

In [5]:
# === WandB Konfiguration ===
use_wandb_widget = widgets.Checkbox(
    value=False,
    description='WandB verwenden',
    style={'description_width': 'initial'}
)

wandb_project_widget = widgets.Text(
    value='gnn_shift_prediction',
    description='WandB Projekt:',
    style={'description_width': 'initial'}
)

# === Grundlegende Parameter ===
seed_widget = widgets.IntText(
    value=0,
    description='Random Seed:',
    style={'description_width': 'initial'}
)

batch_size_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=32,
    description='Batch Size:',
    style={'description_width': 'initial'}
)

num_epochs_widget = widgets.IntSlider(
    value=80,
    min=10,
    max=500,
    step=10,
    description='Epochen:',
    style={'description_width': 'initial'}
)

lr_widget = widgets.FloatLogSlider(
    value=2e-4,
    base=10,
    min=-5,
    max=-2,
    step=0.1,
    description='Learning Rate:',
    style={'description_width': 'initial'},
    readout_format='.2e'
)

# === Modell-Architektur ===
hidden_dim_widget = widgets.Dropdown(
    options=[32, 64, 128, 256, 512],
    value=128,
    description='Hidden Dim:',
    style={'description_width': 'initial'}
)

out_dim_widget = widgets.Dropdown(
    options=[32, 64, 128, 256, 512],
    value=128,
    description='Output Dim:',
    style={'description_width': 'initial'}
)

num_gnn_layers_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=8,
    description='GNN Layers:',
    style={'description_width': 'initial'}
)

operator_type_widget = widgets.Dropdown(
    options=['SAGEConv', 'GCNConv', 'GATConv', 'GATv2Conv', 'GraphConv', 'NNConv', 'GINEConv', 'TransformerConv'],
    value='SAGEConv',
    description='Operator:',
    style={'description_width': 'initial'}
)

# === Dropout ===
encoder_dropout_widget = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.5,
    step=0.05,
    description='Encoder Dropout:',
    style={'description_width': 'initial'}
)

gnn_dropout_widget = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.5,
    step=0.05,
    description='GNN Dropout:',
    style={'description_width': 'initial'}
)

# === Optimizer & Scheduler ===
optimizer_widget = widgets.Dropdown(
    options=['Adam', 'SGD', 'AdamW'],
    value='Adam',
    description='Optimizer:',
    style={'description_width': 'initial'}
)

weight_decay_widget = widgets.FloatLogSlider(
    value=5e-5,
    base=10,
    min=-6,
    max=-2,
    step=0.1,
    description='Weight Decay:',
    style={'description_width': 'initial'},
    readout_format='.2e'
)

scheduler_factor_widget = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=0.9,
    step=0.1,
    description='Scheduler Factor:',
    style={'description_width': 'initial'}
)

scheduler_patience_widget = widgets.IntSlider(
    value=15,
    min=5,
    max=50,
    description='Scheduler Patience:',
    style={'description_width': 'initial'}
)

# === Loss Weights ===
loss_weight_H_widget = widgets.FloatSlider(
    value=10.0,
    min=1.0,
    max=20.0,
    step=1.0,
    description='Loss Weight H:',
    style={'description_width': 'initial'}
)

loss_weight_C_widget = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Loss Weight C:',
    style={'description_width': 'initial'}
)

# === Normalisierung ===
normalize_nodes_widget = widgets.Checkbox(
    value=True,
    description='Node Features normalisieren',
    style={'description_width': 'initial'}
)

normalize_edges_widget = widgets.Checkbox(
    value=True,
    description='Edge Features normalisieren',
    style={'description_width': 'initial'}
)

# === Split Ratio ===
train_ratio_widget = widgets.FloatSlider(
    value=0.825,
    min=0.5,
    max=0.95,
    step=0.025,
    description='Train Ratio:',
    style={'description_width': 'initial'}
)

val_ratio_widget = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=0.3,
    step=0.025,
    description='Val Ratio:',
    style={'description_width': 'initial'}
)

# === Output ===
output_predictions_widget = widgets.Checkbox(
    value=True,
    description='Detaillierte Vorhersagen speichern',
    style={'description_width': 'initial'}
)

output_dir_widget = widgets.Text(
    value='results',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout erstellen
wandb_box = widgets.VBox([
    widgets.HTML('<h4>WandB Konfiguration</h4>'),
    use_wandb_widget,
    wandb_project_widget
])

basic_box = widgets.VBox([
    widgets.HTML('<h4>Grundeinstellungen</h4>'),
    seed_widget,
    batch_size_widget,
    num_epochs_widget,
    lr_widget
])

model_box = widgets.VBox([
    widgets.HTML('<h4>Modell-Architektur</h4>'),
    hidden_dim_widget,
    out_dim_widget,
    num_gnn_layers_widget,
    operator_type_widget,
    encoder_dropout_widget,
    gnn_dropout_widget
])

optim_box = widgets.VBox([
    widgets.HTML('<h4>Optimizer & Scheduler</h4>'),
    optimizer_widget,
    weight_decay_widget,
    scheduler_factor_widget,
    scheduler_patience_widget
])

loss_box = widgets.VBox([
    widgets.HTML('<h4>Loss & Normalisierung</h4>'),
    loss_weight_H_widget,
    loss_weight_C_widget,
    normalize_nodes_widget,
    normalize_edges_widget
])

split_box = widgets.VBox([
    widgets.HTML('<h4>Daten-Split & Output</h4>'),
    train_ratio_widget,
    val_ratio_widget,
    output_predictions_widget,
    output_dir_widget
])

# Alles anzeigen
left_column = widgets.VBox([wandb_box, basic_box, model_box])
right_column = widgets.VBox([optim_box, loss_box, split_box])

display(widgets.HBox([left_column, right_column]))

NameError: name 'widgets' is not defined

### 1.2 Training starten

In [ ]:
def build_config_from_widgets():
    """Erstellt ein Config-Objekt aus den Widget-Werten."""
    class Config:
        pass
    
    config = Config()
    
    # Grundeinstellungen
    config.seed = seed_widget.value
    config.batch_size = batch_size_widget.value
    config.num_epochs = num_epochs_widget.value
    config.lr = lr_widget.value
    
    # Modell-Architektur
    config.hidden_dim = hidden_dim_widget.value
    config.out_dim = out_dim_widget.value
    config.num_gnn_layers = num_gnn_layers_widget.value
    config.operator_type = operator_type_widget.value
    config.encoder_dropout = encoder_dropout_widget.value
    config.gnnlayer_dropout = gnn_dropout_widget.value
    
    # Operator kwargs
    config.operator_kwargs = {}
    if config.operator_type in ['GATConv', 'GATv2Conv']:
        config.operator_kwargs['add_self_loops'] = False
    
    # Optimizer & Scheduler
    config.optimizer = optimizer_widget.value
    config.weight_decay = weight_decay_widget.value
    config.scheduler_factor = scheduler_factor_widget.value
    config.scheduler_patience = scheduler_patience_widget.value
    
    # Loss
    config.loss_weight_H = loss_weight_H_widget.value
    config.loss_weight_C = loss_weight_C_widget.value
    
    # Normalisierung
    config.normalize_node_features = normalize_nodes_widget.value
    config.normalize_edge_features = normalize_edges_widget.value
    
    # Split
    test_ratio = 1.0 - train_ratio_widget.value - val_ratio_widget.value
    config.split_ratio = (train_ratio_widget.value, val_ratio_widget.value, test_ratio)
    
    # Output
    config.output_detailed_predictions = output_predictions_widget.value
    config.output_dir = output_dir_widget.value
    
    return config


def run_training():
    """Führt das Training mit den aktuellen Widget-Einstellungen durch."""
    
    # WandB initialisieren (optional)
    if use_wandb_widget.value:
        import wandb
        wandb.init(project=wandb_project_widget.value)
        config = wandb.config
        # Widget-Werte in wandb.config übertragen
        for key, value in vars(build_config_from_widgets()).items():
            setattr(config, key, value)
    else:
        config = build_config_from_widgets()
    
    # Seed setzen
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training auf: {device}")
    
    # Dataloaders erstellen
    print("Lade Daten...")
    train_loader, val_loader, test_loader = create_dataloaders(
        batch_size=config.batch_size,
        split_ratio=config.split_ratio,
        normalize_node_features=config.normalize_node_features,
        normalize_edge_features=config.normalize_edge_features
    )
    print(f"Train: {len(train_loader)} Batches, Val: {len(val_loader)} Batches, Test: {len(test_loader)} Batches")
    
    # Modell erstellen
    in_dim_dict = {
        "H": 34,
        "C": 39,
        "Others": 16
    }
    
    model = HeteroGNNModel(
        in_dim_dict,
        hidden_dim=config.hidden_dim,
        out_dim=config.out_dim,
        encoder_dropout=config.encoder_dropout,
        gnnlayer_dropout=config.gnnlayer_dropout,
        num_gnn_layers=config.num_gnn_layers,
        operator_type=config.operator_type,
        operator_kwargs=config.operator_kwargs,
        edge_in_dim=10
    ).to(device)
    
    print(f"\nModell erstellt: {config.operator_type} mit {config.num_gnn_layers} Layern")
    
    # Training
    print("\nStarte Training...")
    trained_model = train_model(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        config
    )
    
    # Config speichern für spätere Verwendung
    config_path = MODELS_DIR / 'config.pkl'
    os.makedirs(MODELS_DIR, exist_ok=True)
    with open(config_path, 'wb') as f:
        pickle.dump(config, f)
    print(f"\nKonfiguration gespeichert: {config_path}")
    
    # WandB beenden
    if use_wandb_widget.value:
        wandb.finish()
    
    return trained_model, config


# Training-Button
train_button = widgets.Button(
    description='Training starten',
    button_style='success',
    icon='play'
)

train_output = widgets.Output()

def on_train_click(b):
    with train_output:
        clear_output()
        try:
            global trained_model, training_config
            trained_model, training_config = run_training()
            print("\n✅ Training abgeschlossen!")
        except Exception as e:
            print(f"\n❌ Fehler beim Training: {e}")
            raise

train_button.on_click(on_train_click)

display(train_button)
display(train_output)

---
## 2. Vorhersagen mit trainiertem Modell

Lade ein trainiertes Modell und erstelle Vorhersagen für neue Daten.

### 2.1 Vorhersage Parameter

In [1]:
# Verfügbare Modelle und Daten auflisten
def list_files(directory, extension):
    """Listet Dateien mit bestimmter Endung in einem Verzeichnis."""
    path = Path(directory)
    if path.exists():
        return [f.name for f in path.glob(f'*{extension}')]
    return []

# Widgets für Vorhersage
model_files = list_files(MODELS_DIR, '.pt')
data_files = list_files(DATA_DIR, '.pkl')

pred_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

pred_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

pred_output_widget = widgets.Text(
    value='predictions.csv',
    description='Output Datei:',
    style={'description_width': 'initial'}
)

# Refresh Button
refresh_button = widgets.Button(
    description='Dateien aktualisieren',
    button_style='info',
    icon='refresh'
)

def on_refresh_click(b):
    model_files = list_files(MODELS_DIR, '.pt')
    data_files = list_files(DATA_DIR, '.pkl')
    pred_model_widget.options = model_files if model_files else ['Keine Modelle gefunden']
    pred_data_widget.options = data_files if data_files else ['Keine Daten gefunden']
    # Auch für Explainer aktualisieren
    exp_model_widget.options = model_files if model_files else ['Keine Modelle gefunden']
    exp_data_widget.options = data_files if data_files else ['Keine Daten gefunden']

refresh_button.on_click(on_refresh_click)

display(widgets.VBox([
    widgets.HTML('<h4>Vorhersage Einstellungen</h4>'),
    refresh_button,
    pred_model_widget,
    pred_data_widget,
    pred_output_widget
]))

NameError: name 'MODELS_DIR' is not defined

### 2.2 Vorhersagen erstellen

In [ ]:
def run_prediction():
    """Führt Vorhersagen mit dem ausgewählten Modell durch."""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Pfade
    model_path = MODELS_DIR / pred_model_widget.value
    data_path = DATA_DIR / pred_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'
    
    # Überprüfungen
    for path, name in [(model_path, 'Modell'), (data_path, 'Daten'), 
                       (config_path, 'Config'), (norm_stats_path, 'Norm Stats'),
                       (edge_stats_path, 'Edge Stats')]:
        if not path.exists():
            raise FileNotFoundError(f"{name} nicht gefunden: {path}")
    
    # Config laden
    with open(config_path, 'rb') as f:
        config = pickle.load(f)
    
    # Stats laden
    with open(norm_stats_path, 'rb') as f:
        norm_stats = pickle.load(f)
    with open(edge_stats_path, 'rb') as f:
        edge_stats = pickle.load(f)
    
    # Dataset erstellen
    dataset = ShiftDataset(
        root_dir=str(DATA_DIR),
        file_name=pred_data_widget.value,
        normalize_node_features=config.normalize_node_features,
        normalize_edge_features=config.normalize_edge_features,
        norm_stats=norm_stats,
        **edge_stats
    )
    
    # Modell erstellen und laden
    in_dim_dict = {"H": 34, "C": 39, "Others": 16}
    
    operator_kwargs = {}
    if config.operator_type in ['GATConv', 'GATv2Conv']:
        operator_kwargs['add_self_loops'] = False
    
    model = HeteroGNNModel(
        in_dim_dict,
        hidden_dim=config.hidden_dim,
        out_dim=config.out_dim,
        encoder_dropout=config.encoder_dropout,
        gnnlayer_dropout=config.gnnlayer_dropout,
        num_gnn_layers=config.num_gnn_layers,
        operator_type=config.operator_type,
        operator_kwargs=operator_kwargs,
        edge_in_dim=10
    )
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    print(f"Modell geladen: {pred_model_widget.value}")
    print(f"Daten: {pred_data_widget.value} ({len(dataset)} Graphen)")
    
    results = []
    
    with torch.no_grad():
        for idx in range(len(dataset)):
            nx_g = dataset.nx_graphs[idx]
            compound = nx_g.graph.get("compound", f"unknown_{idx}")
            structure = nx_g.graph.get("structure", "unknown")
            
            data = dataset[idx].to(device)
            
            x_dict = {ntype: data[ntype].x for ntype in data.node_types}
            edge_index_dict = {}
            edge_attr_dict = {}
            for store in data.edge_stores:
                src, rel, dst = store._key
                edge_index_dict[(src, rel, dst)] = store.edge_index
                edge_attr_dict[(src, rel, dst)] = store.edge_attr
            
            out_dict = model(x_dict, edge_index_dict, edge_attr_dict)
            
            # Knoten sammeln
            h_nodes, c_nodes = [], []
            for node in nx_g.nodes():
                element = nx_g.nodes[node]["element"]
                if element == "H":
                    h_nodes.append(node)
                elif element == "C":
                    c_nodes.append(node)
            
            # H-Vorhersagen
            if 'H' in out_dict and out_dict['H'] is not None:
                for i, node in enumerate(h_nodes):
                    attrs = nx_g.nodes[node]
                    shift_low = attrs.get("shift_low", 0.0)
                    prediction = out_dict['H'][i].item() if i < out_dict['H'].shape[0] else float('nan')
                    results.append({
                        'compound': compound,
                        'structure': structure,
                        'atom_type': 'H',
                        'atom_idx': node,
                        'shift_low': shift_low,
                        'prediction': prediction
                    })
            
            # C-Vorhersagen
            if 'C' in out_dict and out_dict['C'] is not None:
                for i, node in enumerate(c_nodes):
                    attrs = nx_g.nodes[node]
                    shift_low = attrs.get("shift_low", 0.0)
                    prediction = out_dict['C'][i].item() if i < out_dict['C'].shape[0] else float('nan')
                    results.append({
                        'compound': compound,
                        'structure': structure,
                        'atom_type': 'C',
                        'atom_idx': node,
                        'shift_low': shift_low,
                        'prediction': prediction
                    })
    
    df = pd.DataFrame(results)
    output_path = PROJECT_ROOT / pred_output_widget.value
    df.to_csv(output_path, index=False)
    
    print(f"\n✅ {len(results)} Vorhersagen gespeichert: {output_path}")
    
    return df


# Prediction Button
predict_button = widgets.Button(
    description='Vorhersagen erstellen',
    button_style='primary',
    icon='calculator'
)

predict_output = widgets.Output()

def on_predict_click(b):
    with predict_output:
        clear_output()
        try:
            global predictions_df
            predictions_df = run_prediction()
            print("\nVorschau der Ergebnisse:")
            display(predictions_df.head(10))
        except Exception as e:
            print(f"\n❌ Fehler bei Vorhersage: {e}")
            raise

predict_button.on_click(on_predict_click)

display(predict_button)
display(predict_output)

---
## 3. GNNExplainer - Erklärungen generieren

Erkläre Vorhersagen für einzelne Knoten mittels GNNExplainer.

### 3.1 Explainer Parameter

In [ ]:
# Explainer Widgets
exp_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

exp_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

exp_graph_idx_widget = widgets.IntText(
    value=0,
    description='Graph Index:',
    style={'description_width': 'initial'}
)

exp_node_type_widget = widgets.Dropdown(
    options=['H', 'C', 'Others'],
    value='H',
    description='Node Type:',
    style={'description_width': 'initial'}
)

exp_node_idx_widget = widgets.IntText(
    value=0,
    description='Node Index:',
    style={'description_width': 'initial'}
)

exp_epochs_widget = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description='Epochen:',
    style={'description_width': 'initial'}
)

exp_lr_widget = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-3,
    max=-1,
    step=0.1,
    description='Learning Rate:',
    style={'description_width': 'initial'},
    readout_format='.3f'
)

exp_type_widget = widgets.Dropdown(
    options=['phenomenon', 'model'],
    value='phenomenon',
    description='Explanation Type:',
    style={'description_width': 'initial'}
)

exp_topk_features_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Features:',
    style={'description_width': 'initial'}
)

exp_topk_edges_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Edges:',
    style={'description_width': 'initial'}
)

exp_output_dir_widget = widgets.Text(
    value='results/explanations',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout
exp_left = widgets.VBox([
    widgets.HTML('<h4>Daten & Modell</h4>'),
    exp_model_widget,
    exp_data_widget,
    exp_graph_idx_widget,
    exp_node_type_widget,
    exp_node_idx_widget
])

exp_right = widgets.VBox([
    widgets.HTML('<h4>Explainer Einstellungen</h4>'),
    exp_epochs_widget,
    exp_lr_widget,
    exp_type_widget,
    exp_topk_features_widget,
    exp_topk_edges_widget,
    exp_output_dir_widget
])

display(widgets.HBox([exp_left, exp_right]))

### 3.2 Erklärung generieren

In [ ]:
def run_explanation():
    """Generiert eine GNNExplainer Erklärung für den ausgewählten Knoten."""
    
    device = get_device(None)
    
    # Pfade
    model_path = MODELS_DIR / exp_model_widget.value
    data_path = DATA_DIR / exp_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'
    
    # Config und Stats laden
    config = load_config(str(config_path))
    norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))
    
    # Dataset erstellen
    dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    validate_indices(len(dataset), exp_graph_idx_widget.value, "graph_idx")
    
    # Daten laden
    data = dataset[exp_graph_idx_widget.value].to(device)
    x_dict, edge_index_dict, edge_attr_dict, y_dict = heterodata_to_dicts(data)
    
    # Modell laden
    base_model = load_trained_model(str(model_path), config, device)
    wrapped_model = NodeTypeRegressionWrapper(base_model, exp_node_type_widget.value)
    
    # Target vorbereiten
    target = None
    target_tensor = y_dict.get(exp_node_type_widget.value)
    
    if target_tensor is not None:
        target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
        validate_indices(target_tensor.size(0), exp_node_idx_widget.value, "node_idx")
        if exp_type_widget.value == "phenomenon":
            if torch.isnan(target_tensor[exp_node_idx_widget.value]):
                raise ValueError(
                    f"Target für Knoten {exp_node_idx_widget.value} ist NaN. "
                    "Wähle einen anderen Knoten oder nutze 'model' als Explanation Type."
                )
            target = target_tensor
    else:
        validate_indices(x_dict[exp_node_type_widget.value].size(0), exp_node_idx_widget.value, "node_idx")
        if exp_type_widget.value == "phenomenon":
            raise ValueError(
                f"Keine Targets für Node Type {exp_node_type_widget.value}; "
                "kann phenomenon nicht erklären."
            )
    
    # Explainer konfigurieren
    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )
    
    explainer = Explainer(
        model=wrapped_model,
        algorithm=GNNExplainer(epochs=exp_epochs_widget.value, lr=exp_lr_widget.value),
        explanation_type=exp_type_widget.value,
        model_config=model_config,
        node_mask_type="attributes",
        edge_mask_type="object",
    )
    
    print(f"Generiere Erklärung für {exp_node_type_widget.value}[{exp_node_idx_widget.value}]...")
    
    # Erklärung generieren
    explanation = explainer(
        x_dict,
        edge_index_dict,
        edge_attr_dict=edge_attr_dict,
        target=target,
        index=exp_node_idx_widget.value,
    )
    
    if not isinstance(explanation, HeteroExplanation):
        raise TypeError(
            f"Erwartete HeteroExplanation, erhielt {type(explanation)}. "
            "Stelle sicher, dass eine aktuelle torch_geometric Version verwendet wird."
        )
    
    # Vorhersage holen
    with torch.no_grad():
        predictions = base_model(x_dict, edge_index_dict, edge_attr_dict)
        node_prediction = float(predictions[exp_node_type_widget.value][exp_node_idx_widget.value].item())
    
    # Zusammenfassungen erstellen
    feature_summary = summarize_node_mask(
        explanation.node_mask_dict,
        exp_node_type_widget.value,
        exp_node_idx_widget.value,
        top_k=max(exp_topk_features_widget.value, 0),
    )
    
    edge_summary = summarize_incident_edges(
        explanation.edge_mask_dict,
        edge_index_dict,
        exp_node_type_widget.value,
        exp_node_idx_widget.value,
        top_k=max(exp_topk_edges_widget.value, 0),
    )
    
    target_value = float(target[exp_node_idx_widget.value].item()) if target is not None else None
    
    summary = {
        "graph_idx": exp_graph_idx_widget.value,
        "node_type": exp_node_type_widget.value,
        "node_idx": exp_node_idx_widget.value,
        "prediction": node_prediction,
        "target": target_value,
        "top_features": feature_summary,
        "important_edges": edge_summary,
    }
    
    # Speichern
    output_dir = PROJECT_ROOT / exp_output_dir_widget.value
    ensure_dir(str(output_dir))
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = f"gnnexplainer_{exp_node_type_widget.value}_n{exp_node_idx_widget.value}_g{exp_graph_idx_widget.value}_{timestamp}"
    
    torch.save(explanation, output_dir / f"{base_name}.pt")
    with open(output_dir / f"{base_name}.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)
    
    print(f"\n✅ Erklärung gespeichert in: {output_dir}")
    
    return summary, explanation


# Explain Button
explain_button = widgets.Button(
    description='Erklärung generieren',
    button_style='warning',
    icon='lightbulb'
)

explain_output = widgets.Output()

def on_explain_click(b):
    with explain_output:
        clear_output()
        try:
            global explanation_summary, explanation_obj
            explanation_summary, explanation_obj = run_explanation()
            print("\n" + "="*50)
            print("ERKLÄRUNGSZUSAMMENFASSUNG")
            print("="*50)
            print(json.dumps(explanation_summary, indent=2))
        except Exception as e:
            print(f"\n❌ Fehler bei Erklärung: {e}")
            raise

explain_button.on_click(on_explain_click)

display(explain_button)
display(explain_output)

---
## 4. Hilfsfunktionen

In [ ]:
def show_dataset_info(data_file):
    """Zeigt Informationen über einen Datensatz an."""
    data_path = DATA_DIR / data_file
    
    if not data_path.exists():
        print(f"Datei nicht gefunden: {data_path}")
        return
    
    # Config laden falls vorhanden
    config_path = MODELS_DIR / 'config.pkl'
    if config_path.exists():
        with open(config_path, 'rb') as f:
            config = pickle.load(f)
        normalize_nodes = config.normalize_node_features
        normalize_edges = config.normalize_edge_features
    else:
        normalize_nodes = False
        normalize_edges = False
    
    # Dataset laden
    dataset = ShiftDataset(
        root_dir=str(DATA_DIR),
        file_name=data_file,
        normalize_node_features=normalize_nodes,
        normalize_edge_features=normalize_edges
    )
    
    print(f"Datensatz: {data_file}")
    print(f"Anzahl Graphen: {len(dataset)}")
    
    if len(dataset) > 0:
        sample = dataset[0]
        print(f"\nBeispiel Graph (Index 0):")
        print(f"  Node Types: {sample.node_types}")
        for ntype in sample.node_types:
            print(f"    {ntype}: {sample[ntype].x.shape[0]} Knoten, {sample[ntype].x.shape[1]} Features")
        print(f"  Edge Types: {len(sample.edge_types)}")


# Widget für Dataset Info
info_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Datensatz:',
    style={'description_width': 'initial'}
)

info_button = widgets.Button(
    description='Info anzeigen',
    button_style='',
    icon='info'
)

info_output = widgets.Output()

def on_info_click(b):
    with info_output:
        clear_output()
        show_dataset_info(info_data_widget.value)

info_button.on_click(on_info_click)

display(widgets.VBox([
    widgets.HTML('<h4>Datensatz-Informationen</h4>'),
    info_data_widget,
    info_button,
    info_output
]))